# Milestone 1: Qualitative Evaluation

The purpose of this section is to compare the performance of the two retrieval approaches developed in this milestone:

- **BM25**, which relies on lexical overlap between the query and indexed documents
- **Semantic search**, which retrieves documents based on embedding similarity

Rather than computing formal retrieval metrics at this stage, the evaluation is qualitative. The aim is to examine the kinds of queries each method handles well, identify their weaknesses, and assess whether the returned results are useful for the underlying user intent.

This evaluation proceeds in two parts:

1. **Create a query set** containing a range of query types and difficulty levels
2. **Retrieve the top 5 results** for each query using both BM25 and semantic search

The resulting outputs will later be compared in detail to understand how the two methods differ in practice.

## Imports

In [ ]:
from pathlib import Path
import pandas as pd

from src.bm25 import BM25Retriever
from src.semantic import SemanticRetriever

from src.preprocessing import find_repo_root

## Create a Query Set

 A set of 10 queries is constructed to evaluate both retrieval systems on a shared input set.

The queries are intentionally designed to vary in difficulty:

- **Easy queries** are short and keyword-oriented. These are expected to favor BM25 because they contain direct lexical signals.
- **Medium queries** are still relatively clear, but less likely to match documents through exact wording alone. These provide a better test of semantic retrieval.
- **Complex queries** express a fuller user intent and may require reasoning beyond direct keyword matching. These are useful for identifying where both retrieval approaches still struggle.

Because the selected category for Milestone 1 is **All_Beauty**, the queries are written to reflect realistic beauty and personal care search behavior.

In [7]:
queries_df = pd.DataFrame([
    {"query_id": 1, "query": "lip balm", "difficulty": "easy"},
    {"query_id": 2, "query": "face moisturizer", "difficulty": "easy"},
    {"query_id": 3, "query": "sunscreen for face", "difficulty": "easy"},
    {"query_id": 4, "query": "something for dry skin", "difficulty": "medium"},
    {"query_id": 5, "query": "product to reduce frizzy hair", "difficulty": "medium"},
    {"query_id": 6, "query": "gentle makeup remover", "difficulty": "medium"},
    {"query_id": 7, "query": "beauty product that is easy to carry while traveling", "difficulty": "complex"},
    {"query_id": 8, "query": "makeup remover that does not irritate sensitive skin", "difficulty": "complex"},
    {"query_id": 9, "query": "skin care product for very dry lips in winter", "difficulty": "complex"},
    {"query_id": 10, "query": "lightweight product that keeps skin hydrated all day", "difficulty": "complex"},
])

queries_df

,query_id,query,difficulty
0,1,lip balm,easy
1,2,face moisturizer,easy
2,3,sunscreen for face,easy
3,4,something for dry skin,medium
4,5,product to reduce frizzy hair,medium
5,6,gentle makeup remover,medium
6,7,beauty product that is easy to carry while tra...,complex
7,8,makeup remover that does not irritate sensitiv...,complex
8,9,skin care product for very dry lips in winter,complex
9,10,lightweight product that keeps skin hydrated a...,complex


The query set covers a useful range of retrieval scenarios. The easy queries should help confirm that the BM25 retriever is functioning as expected for exact or near-exact lexical matches. The medium and complex queries are more informative because they test whether semantic similarity can recover relevant items even when the wording in the query differs from the wording in the indexed documents.

This design makes it possible to compare not only which method performs better overall, but also **which method performs better for different query types**.

## Retrieve Results

Each query is now run through both retrieval systems:

- **BM25 retriever**
- **Semantic retriever**

For each method, the **top 5 results** are collected.

The purpose of this step is not yet to judge which method is better, but to assemble a consistent set of outputs that can later be compared query by query. Each result includes the document identifier, title, rating, retrieval score, and cleaned retrieval text. These fields are sufficient for both inspection and later reporting.

In [13]:
repo_root = find_repo_root()
data_path = repo_root / Path("data/processed/All_Beauty_clean.parquet")
df = pd.read_parquet(data_path)
documents = df.to_dict(orient="records")

In [ ]:
# Set up directories for loading retrievers
bm25_dir = repo_root / Path("data/processed/bm25_index")
semantic_dir = repo_root / Path("data/processed/semantic_index")

In [ ]:
# BM25: load if artifacts exist, otherwise build and save
if (bm25_dir / "bm25_documents.pkl").exists() and \
   (bm25_dir / "bm25_tokenized_corpus.pkl").exists() and \
   (bm25_dir / "bm25_index.pkl").exists():
    bm25 = BM25Retriever.load(bm25_dir)
else:
    bm25 = BM25Retriever(documents)
    bm25.save(bm25_dir)

# Semantic: load if artifacts exist, otherwise build and save
if (semantic_dir / "semantic_documents.pkl").exists() and \
   (semantic_dir / "semantic_model_name.json").exists() and \
   (semantic_dir / "semantic_embeddings.npy").exists() and \
   (semantic_dir / "semantic_faiss.index").exists():
    semantic = SemanticRetriever.load(semantic_dir)
else:
    semantic = SemanticRetriever(documents)
    semantic.save(semantic_dir)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2935.05it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 21910/21910 [1:47:47<00:00,  3.39it/s]   


To make the comparison systematic, a helper function is defined below. It runs a retriever over the full query set and stores the outputs in a structured table. This makes it easier to inspect results, save them for reporting, and compare methods side by side.

In [17]:
def collect_results(
    retriever,
    queries_df: pd.DataFrame,
    method_name: str,
    top_k: int = 5,
) -> pd.DataFrame:
    """
    Collect top-k retrieval results for a set of queries.

    Parameters
    ----------
    retriever : object
        Retriever instance with a `search(query, top_k=...)` method.
    queries_df : pd.DataFrame
        DataFrame containing `query_id`, `query`, and `difficulty`.
    method_name : str
        Name of retrieval method, such as "BM25" or "Semantic".
    top_k : int, default=5
        Number of results to collect per query.

    Returns
    -------
    pd.DataFrame
        A long-format DataFrame containing one row per retrieved result.
    """
    rows = []

    for _, row in queries_df.iterrows():
        query_id = row["query_id"]
        query = row["query"]
        difficulty = row["difficulty"]

        results = retriever.search(query, top_k=top_k)

        for rank, result in enumerate(results, start=1):
            rows.append(
                {
                    "method": method_name,
                    "query_id": query_id,
                    "query": query,
                    "difficulty": difficulty,
                    "rank": rank,
                    "doc_id": result.get("doc_id"),
                    "title": result.get("title"),
                    "rating": result.get("rating"),
                    "score": result.get("score"),
                    "text": result.get("text"),
                }
            )

    return pd.DataFrame(rows)

In [18]:
# Code cell: collect both result sets
bm25_results = collect_results(
    retriever=bm25,
    queries_df=queries_df,
    method_name="BM25",
    top_k=5,
)

semantic_results = collect_results(
    retriever=semantic,
    queries_df=queries_df,
    method_name="Semantic",
    top_k=5,
)

all_results = pd.concat([bm25_results, semantic_results], ignore_index=True)
all_results.head(10)

,method,query_id,query,difficulty,rank,doc_id,title,rating,score,text
0,BM25,1,lip balm,easy,1,B01C052LVS_547994,best lip balm,5,20.889257,best lip balm best lip balm ever!
1,BM25,1,lip balm,easy,2,B01N5CJPSX_43223,Best lip balm ever,5,20.650096,best lip balm ever best lip balm ever
2,BM25,1,lip balm,easy,3,B01NAMCYEI_78219,beautiful lip balm,5,20.650096,beautiful lip balm great price great lip balm
3,BM25,1,lip balm,easy,4,B005JHJUI2_111547,Great daily lip balm!,5,20.650096,great daily lip balm! my fav lip balm!
4,BM25,1,lip balm,easy,5,B01MZ2ZBOJ_78218,PRETTY LIP BALM,5,20.416349,pretty lip balm very pretty and inexpensive li...
5,BM25,2,face moisturizer,easy,1,B01MRR1SAS_591326,Nice lightweight face moisturizer,5,14.030204,nice lightweight face moisturizer nice lightwe...
6,BM25,2,face moisturizer,easy,2,B007HLWRTM_154324,Great moisturizer!,5,13.844107,great moisturizer! great moisturizer that does...
7,BM25,2,face moisturizer,easy,3,B07X9VJCN9_622049,Great Face Moisturizer!,4,13.731127,great face moisturizer! good! this face moistu...
8,BM25,2,face moisturizer,easy,4,B07MX673RB_241638,Good product,5,13.658535,good product face moisturizer
9,BM25,2,face moisturizer,easy,5,B08XMBHL8C_92790,Moisturizer,3,13.508304,moisturizer love the face moisturizer but this...


The table above stores the retrieved results in long format, with one row per query-result pair. This makes the output easier to sort, filter, and compare across retrieval methods.

Before writing the qualitative discussion, the retrieved results need to be examined in a more interpretable form. Since the full results table contains outputs for all queries and both retrieval methods, the following helper function is used to isolate the results for one query at a time. This makes it easier to compare BM25 and semantic retrieval directly for the same information need.

In [19]:
def preview_results_for_query(results_df: pd.DataFrame, query_id: int) -> pd.DataFrame:
    """
    Return formatted retrieval results for a single query.

    Parameters
    ----------
    results_df : pd.DataFrame
        Combined retrieval results.
    query_id : int
        Query identifier to filter by.

    Returns
    -------
    pd.DataFrame
        A filtered DataFrame for the selected query.
    """
    cols = ["method", "rank", "query", "title", "rating", "score", "text"]
    return (
        results_df.loc[results_df["query_id"] == query_id, cols]
        .sort_values(["method", "rank"])
        .reset_index(drop=True)
    )

In [20]:
preview_results_for_query(all_results, query_id=1)

,method,rank,query,title,rating,score,text
0,BM25,1,lip balm,best lip balm,5,20.889257,best lip balm best lip balm ever!
1,BM25,2,lip balm,Best lip balm ever,5,20.650096,best lip balm ever best lip balm ever
2,BM25,3,lip balm,beautiful lip balm,5,20.650096,beautiful lip balm great price great lip balm
3,BM25,4,lip balm,Great daily lip balm!,5,20.650096,great daily lip balm! my fav lip balm!
4,BM25,5,lip balm,PRETTY LIP BALM,5,20.416349,pretty lip balm very pretty and inexpensive li...
5,Semantic,1,lip balm,Lip balm,5,0.893845,lip balm awesome..
6,Semantic,2,lip balm,The best lip balm to exist.,5,0.872642,the best lip balm to exist. seriously
7,Semantic,3,lip balm,Great lip balm,5,0.857406,great lip balm my wife loves this stuff.
8,Semantic,4,lip balm,My fav lip balm,5,0.853007,"my fav lip balm love this product, second time..."
9,Semantic,5,lip balm,Very good lip balm. Too expensive.,5,0.846349,very good lip balm. too expensive. good produc...


## Save outputs for later discussion

To support the later discussion in `results/milestone1_discussion.md`, the retrieval outputs are saved to disk. This ensures that the comparison can be revisited without rerunning the full notebook.

In [21]:
output_path = Path("results/retrieval_results.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

all_results.to_csv(output_path, index=False)
print(f"Saved retrieval results to: {output_path}")

Saved retrieval results to: results\retrieval_results.csv


At this point, both retrieval methods have been evaluated on the same 10-query set, and their top 5 outputs have been collected in a structured format.

These results provide the basis for the next part of the milestone, where selected queries will be compared in detail. That comparison will focus on relevance, usefulness, failure cases, and differences in behavior between keyword-based and embedding-based retrieval.